In [2]:
import mne
import numpy as np

In [4]:
# Load files
psg = mne.io.read_raw_edf("ST7011J0-PSG.edf", preload=True)
annotations = mne.read_annotations("ST7011JP-Hypnogram.edf")

# Attach labels
psg.set_annotations(annotations)

# Convert annotations → events
events, event_id = mne.events_from_annotations(psg)

# Epoch into 30s windows
epochs = mne.Epochs(
    psg,
    events,
    event_id,
    tmin=0,
    tmax=30,
    baseline=None,
    preload=True
)

X = epochs.get_data()
y = epochs.events[:, -1]

print(X.shape, y.shape)

Extracting EDF parameters from ST7011J0-PSG.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 3589999  =      0.000 ... 35899.990 secs...


/var/folders/jc/lnxp8vkd5858vcd9vjh2hqn80000gn/T/ipykernel_29182/1460319443.py:2: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  psg = mne.io.read_raw_edf("ST7011J0-PSG.edf", preload=True)
/var/folders/jc/lnxp8vkd5858vcd9vjh2hqn80000gn/T/ipykernel_29182/1460319443.py:2: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  psg = mne.io.read_raw_edf("ST7011J0-PSG.edf", preload=True)


Used Annotations descriptions: ['Sleep stage 1', 'Sleep stage 2', 'Sleep stage 3', 'Sleep stage 4', 'Sleep stage R', 'Sleep stage W']
Not setting metadata
231 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 231 events and 3001 original time points ...
0 bad epochs dropped
(231, 5, 3001) (231,)


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# flatten
X_flat = X.reshape(X.shape[0], -1)

X_train, X_test, y_train, y_test = train_test_split(X_flat, y, test_size=0.2)

clf = RandomForestClassifier(n_estimators=100)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.78      0.58      0.67        12
           2       0.44      0.88      0.58        16
           3       0.00      0.00      0.00        13
           4       0.00      0.00      0.00         3
           5       0.00      0.00      0.00         1
           6       0.50      0.50      0.50         2

    accuracy                           0.47        47
   macro avg       0.29      0.33      0.29        47
weighted avg       0.37      0.47      0.39        47



/Users/ethanfreshman/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/ethanfreshman/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/ethanfreshman/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


## CNN-based classification (baseline)

In [ ]:
# CNN-based signal classification (baseline)
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv1D, BatchNormalization, ReLU, MaxPooling1D, Dropout, GlobalAveragePooling1D, Dense
from tensorflow.keras.callbacks import EarlyStopping

# Map EDF annotation descriptions to canonical sleep stages
stage_map = {
    "Sleep stage W": "W",
    "Sleep stage 1": "N1",
    "Sleep stage 2": "N2",
    "Sleep stage 3": "N3",
    "Sleep stage 4": "N3",  # merge legacy stage 4 into N3
    "Sleep stage R": "REM",
}

# Build integer-event -> description lookup from event_id (created in cell 2)
inv_event_id = {v: k for k, v in event_id.items()}
raw_desc = np.array([inv_event_id.get(code, "UNKNOWN") for code in y])
stage_text = np.array([stage_map.get(desc, "OTHER") for desc in raw_desc])

# Keep only target stages (drop movement/unknown/etc.)
mask = stage_text != "OTHER"
X_stage = X[mask]
y_stage = stage_text[mask]

if len(X_stage) == 0:
    raise ValueError("No usable sleep-stage epochs found after label filtering.")

# Conv1D expects (samples, timesteps, channels)
X_cnn = np.transpose(X_stage, (0, 2, 1)).astype(np.float32)

# Per-channel z-score normalization for stable training
mean = X_cnn.mean(axis=(0, 1), keepdims=True)
std = X_cnn.std(axis=(0, 1), keepdims=True) + 1e-8
X_cnn = (X_cnn - mean) / std

# Encode class labels to integers
le = LabelEncoder()
y_enc = le.fit_transform(y_stage)
class_names = le.classes_

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_cnn, y_enc,
    test_size=0.2,
    random_state=42,
    stratify=y_enc,
)

# Compute class weights to reduce class-imbalance bias
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}

# Build a compact 1D CNN baseline
n_timesteps, n_channels = X_train.shape[1], X_train.shape[2]
n_classes = len(class_names)

model = Sequential([
    Conv1D(32, kernel_size=7, padding="same", input_shape=(n_timesteps, n_channels)),
    BatchNormalization(),
    ReLU(),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    Conv1D(64, kernel_size=5, padding="same"),
    BatchNormalization(),
    ReLU(),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),

    Conv1D(128, kernel_size=3, padding="same"),
    BatchNormalization(),
    ReLU(),
    GlobalAveragePooling1D(),

    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(n_classes, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1,
    shuffle=True,
)

# Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")
print("\nClass order:", list(class_names))
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))